# DaTSCAN — Fase 8: comparación final y blending OOF cruzado

Este notebook determina si la CNN local estratificada por protocolo aporta información complementaria al mejor modelo OOF anterior.

Principios:

1. solo se aceptan predicciones realmente *out-of-fold*;
2. todos los archivos se alinean por `uid`, nunca por orden de fila;
3. el peso del ensamble se elige con los otros cuatro meta-folds y se aplica al fold excluido;
4. se comparan mezcla de probabilidades y mezcla en escala logit;
5. la decisión exige una mejora cruzada mínima y un intervalo *bootstrap* razonable;
6. no se entrena otra CNN.

La fase termina aquí aunque el ensamble no mejore: un resultado negativo también decide que debe conservarse el modelo de referencia.

## 0. Dependencias

In [ ]:
# Descomente solamente si falta alguna dependencia.
# %pip install numpy pandas scipy scikit-learn matplotlib seaborn

## 1. Librerías y configuración

In [ ]:
from pathlib import Path
import os
import json, warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.special import expit, logit
from sklearn.metrics import log_loss, roc_auc_score, brier_score_loss
from IPython.display import display

warnings.filterwarnings('ignore',category=FutureWarning);sns.set_theme(style='whitegrid')
SEED=20260910;EPS=1e-6;N_BOOTSTRAP=2000;MIN_ACCEPTED_GAIN=.003
rng=np.random.default_rng(SEED)

In [ ]:
DATA_ROOT=Path(r'C:\Users\DELL\OneDrive\Escritorio\kaggle\parkinson')
PROJECT_DIR=DATA_ROOT/'latent_protocol_cv'
CURRENT_OOF=PROJECT_DIR/'cv_protocol_stratified_2ch_v1'/'protocol_stratified_oof.csv'
FOLDS_CSV=PROJECT_DIR/'cv_protocol_stratified_2ch_v1'/'train_protocol_stratified_folds.csv'
OUTPUT_DIR=PROJECT_DIR/'blending_oof_final_v1'
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)

# Si conoce la ruta exacta del mejor OOF anterior, escríbala aquí.
# Ejemplo: REFERENCE_OOF=DATA_ROOT/'artifacts'/'modelo_x'/'oof_predictions.csv'
REFERENCE_OOF=None
APPROVE_REFERENCE=False

for p in (CURRENT_OOF,FOLDS_CSV):
    print(p,'| existe:',p.exists())
    if not p.exists():raise FileNotFoundError(p)
print('Salida:',OUTPUT_DIR)

## 2. Funciones para reconocer y normalizar archivos OOF

In [ ]:
UID_CANDIDATES=('uid','study_uid','studyinstanceuid','id')
TARGET_CANDIDATES=('target','is_pathologic','label','y','truth')
PREDICTION_CANDIDATES=('prediction','probability','prob','pred','oof_prediction','oof_pred','oof_probability',
                       'ensemble_prediction','ensemble_probability','oof_ensemble','is_pathologic_probability')

def detect_column(columns,candidates):
    mapping={str(c).lower():c for c in columns}
    return next((mapping[c.lower()] for c in candidates if c.lower() in mapping),None)

def read_oof(path,require_target=True):
    frame=pd.read_csv(path);uid_col=detect_column(frame.columns,UID_CANDIDATES)
    target_col=detect_column(frame.columns,TARGET_CANDIDATES);pred_col=detect_column(frame.columns,PREDICTION_CANDIDATES)
    if pred_col is None:
        possible=[]
        for c in frame.columns:
            if c in (uid_col,target_col):continue
            values=pd.to_numeric(frame[c],errors='coerce')
            if values.notna().all() and values.between(0,1).all() and any(token in str(c).lower() for token in ('pred','prob','oof','ensemble')):
                possible.append(c)
        if len(possible)==1:pred_col=possible[0]
    if uid_col is None or pred_col is None or (require_target and target_col is None):
        raise KeyError(f'No se detectaron uid/target/prediction en {path.name}')
    keep=[uid_col]+([target_col] if target_col is not None else[])+[pred_col]
    out=frame[keep].copy();out.columns=['uid']+(['target'] if target_col is not None else[])+['prediction']
    out.uid=out.uid.astype(str).str.strip();out.prediction=pd.to_numeric(out.prediction,errors='coerce')
    if 'target' in out:out.target=pd.to_numeric(out.target,errors='coerce')
    if out.uid.duplicated().any():raise ValueError(f'UID duplicados: {path}')
    if out.prediction.isna().any() or not out.prediction.between(0,1).all():raise ValueError(f'Predicciones inválidas: {path}')
    return out

def metrics(y,p):
    y=np.asarray(y,dtype=int);p=np.clip(np.asarray(p,float),EPS,1-EPS)
    return {'n':len(y),'logloss':log_loss(y,p,labels=[0,1]),'auc':roc_auc_score(y,p),'brier':brier_score_loss(y,p)}

## 3. Inventario automático de candidatos OOF

El inventario es una ayuda, no una autorización. Solo considera CSV cuyo nombre contiene `oof`, que poseen 1,362 UID únicos, una etiqueta binaria y una columna de probabilidad reconocible. Revise la ruta: predicciones generadas sobre los mismos pacientes usados para ajustar el modelo no son OOF válidas.

In [ ]:
current=read_oof(CURRENT_OOF)
expected_uids=set(current.uid);candidate_rows=[]
for path in DATA_ROOT.rglob('*.csv'):
    if 'oof' not in path.name.lower() or path.resolve()==CURRENT_OOF.resolve():continue
    try:
        frame=read_oof(path)
        if len(frame)!=len(current) or set(frame.uid)!=expected_uids:continue
        if set(frame.target.dropna().astype(int).unique())-{0,1}:continue
        m=metrics(frame.target,frame.prediction)
        candidate_rows.append({'path':str(path.resolve()),**m})
    except Exception:pass
candidates=pd.DataFrame(candidate_rows)
if len(candidates):candidates=candidates.sort_values('logloss').reset_index(drop=True)
display(candidates)
candidates.to_csv(OUTPUT_DIR/'oof_candidate_inventory.csv',index=False)
if REFERENCE_OOF is None:
    print('Seleccione una ruta válida de la tabla y asígnela manualmente a REFERENCE_OOF.')
else:print('Referencia configurada:',REFERENCE_OOF)

## 4. Selección explícita del modelo de referencia

Copie la ruta exacta del mejor OOF válido. Después cambie `APPROVE_REFERENCE=True`. No seleccione automáticamente el menor resultado sin confirmar que cada predicción fue obtenida con el paciente fuera del entrenamiento.

In [ ]:
# Ejemplo después de revisar la tabla:
# REFERENCE_OOF=Path(r'C:\...\mejor_modelo\oof_predictions.csv')
# APPROVE_REFERENCE=True

if REFERENCE_OOF is None:raise RuntimeError('Configure REFERENCE_OOF con la ruta de un OOF válido.')
REFERENCE_OOF=Path(REFERENCE_OOF)
if not REFERENCE_OOF.exists():raise FileNotFoundError(REFERENCE_OOF)
if not APPROVE_REFERENCE:raise RuntimeError('Revise la procedencia del archivo y cambie APPROVE_REFERENCE=True.')

## 5. Alineación por UID y verificaciones

In [ ]:
reference=read_oof(REFERENCE_OOF).rename(columns={'prediction':'prediction_reference','target':'target_reference'})
current_named=current.rename(columns={'prediction':'prediction_current','target':'target_current'})
folds=pd.read_csv(FOLDS_CSV)
fold_col='protocol_stratified_fold'
required_fold={'uid','target','protocol_cluster',fold_col}
if required_fold-set(folds.columns):raise KeyError(f'Faltan columnas: {sorted(required_fold-set(folds.columns))}')

data=(folds[['uid','target','protocol_cluster',fold_col]]
      .merge(reference,on='uid',validate='one_to_one')
      .merge(current_named,on='uid',validate='one_to_one'))
if len(data)!=1362:raise ValueError(f'La intersección contiene {len(data)} casos, no 1,362.')
if not data.target.eq(data.target_reference.astype(int)).all():raise ValueError('Etiquetas incompatibles con la referencia.')
if not data.target.eq(data.target_current.astype(int)).all():raise ValueError('Etiquetas incompatibles con la CNN actual.')
data=data.drop(columns=['target_reference','target_current'])
for c in ('prediction_reference','prediction_current'):data[c]=data[c].clip(EPS,1-EPS)
print('Alineación correcta:',len(data),'UID')
display(data.head())

## 6. Comparación individual y complementariedad

In [ ]:
individual=pd.DataFrame([
    {'model':'Referencia',**metrics(data.target,data.prediction_reference)},
    {'model':'CNN local estratificada',**metrics(data.target,data.prediction_current)}])
display(individual);individual.to_csv(OUTPUT_DIR/'individual_metrics.csv',index=False)

data['error_reference']=data.target-data.prediction_reference
data['error_current']=data.target-data.prediction_current
correlations=pd.Series({
    'probability_pearson':data.prediction_reference.corr(data.prediction_current),
    'probability_spearman':data.prediction_reference.corr(data.prediction_current,method='spearman'),
    'error_pearson':data.error_reference.corr(data.error_current),
    'error_spearman':data.error_reference.corr(data.error_current,method='spearman')},name='correlation').to_frame()
display(correlations);correlations.to_csv(OUTPUT_DIR/'prediction_error_correlations.csv')

plt.figure(figsize=(6,5));sns.scatterplot(data=data,x='prediction_reference',y='prediction_current',hue='target',alpha=.45,s=20)
plt.plot([0,1],[0,1],'--',color='gray');plt.tight_layout();plt.savefig(OUTPUT_DIR/'prediction_scatter.png',dpi=170,bbox_inches='tight');plt.show()

## 7. Selección cruzada del peso

`w` es el peso del modelo de referencia. En cada meta-fold se selecciona el peso usando los otros cuatro folds. Se comparan:

- mezcla lineal de probabilidades;
- mezcla lineal de logits, normalmente más apropiada para log-loss.

In [ ]:
WEIGHTS=np.linspace(0,1,101)
def blend_predictions(pref,pcur,w,kind):
    if kind=='probability':return np.clip(w*pref+(1-w)*pcur,EPS,1-EPS)
    if kind=='logit':return expit(w*logit(np.clip(pref,EPS,1-EPS))+(1-w)*logit(np.clip(pcur,EPS,1-EPS)))
    raise ValueError(kind)

for kind in ('probability','logit'):data[f'blend_{kind}_cf']=np.nan
weight_rows=[]
for kind in ('probability','logit'):
    for heldout in sorted(data[fold_col].unique()):
        fit=data[fold_col].ne(heldout);apply=data[fold_col].eq(heldout)
        losses=[]
        for w in WEIGHTS:
            p=blend_predictions(data.loc[fit,'prediction_reference'].to_numpy(),data.loc[fit,'prediction_current'].to_numpy(),w,kind)
            losses.append(log_loss(data.loc[fit,'target'],p,labels=[0,1]))
        best_index=int(np.argmin(losses));best_w=float(WEIGHTS[best_index])
        data.loc[apply,f'blend_{kind}_cf']=blend_predictions(data.loc[apply,'prediction_reference'].to_numpy(),data.loc[apply,'prediction_current'].to_numpy(),best_w,kind)
        weight_rows.append({'kind':kind,'heldout_fold':heldout,'weight_reference':best_w,'fit_logloss':losses[best_index],'n_fit':int(fit.sum()),'n_apply':int(apply.sum())})
weights=pd.DataFrame(weight_rows);display(weights);weights.to_csv(OUTPUT_DIR/'cross_fitted_weights.csv',index=False)
if data[['blend_probability_cf','blend_logit_cf']].isna().any().any():raise RuntimeError('Blending cruzado incompleto.')

## 8. Métricas cruzadas del ensamble

In [ ]:
results=pd.DataFrame([
    {'model':'Referencia',**metrics(data.target,data.prediction_reference)},
    {'model':'CNN local',**metrics(data.target,data.prediction_current)},
    {'model':'Blend probabilidades CF',**metrics(data.target,data.blend_probability_cf)},
    {'model':'Blend logits CF',**metrics(data.target,data.blend_logit_cf)}])
reference_loss=float(results.loc[results.model.eq('Referencia'),'logloss'].iloc[0])
results['gain_vs_reference']=reference_loss-results.logloss
display(results.sort_values('logloss'));results.to_csv(OUTPUT_DIR/'cross_fitted_blend_metrics.csv',index=False)

## 9. Comparación por protocolo

In [ ]:
rows=[]
models={'Referencia':'prediction_reference','CNN local':'prediction_current','Blend probabilidades CF':'blend_probability_cf','Blend logits CF':'blend_logit_cf'}
for cluster,part in data.groupby('protocol_cluster'):
    for name,col in models.items():rows.append({'protocol_cluster':cluster,'model':name,**metrics(part.target,part[col])})
by_cluster=pd.DataFrame(rows);display(by_cluster);by_cluster.to_csv(OUTPUT_DIR/'blend_metrics_by_cluster.csv',index=False)
fig,axes=plt.subplots(1,2,figsize=(14,4.5))
sns.barplot(data=by_cluster,x='protocol_cluster',y='logloss',hue='model',ax=axes[0])
sns.barplot(data=by_cluster,x='protocol_cluster',y='auc',hue='model',ax=axes[1])
for ax in axes:ax.legend(fontsize=7)
plt.tight_layout();plt.savefig(OUTPUT_DIR/'blend_by_cluster.png',dpi=170,bbox_inches='tight');plt.show()

## 10. Incertidumbre mediante bootstrap pareado

In [ ]:
best_blend_row=results[results.model.str.startswith('Blend')].sort_values('logloss').iloc[0]
best_blend_col='blend_probability_cf' if best_blend_row.model=='Blend probabilidades CF' else 'blend_logit_cf'
y=data.target.to_numpy();pref=data.prediction_reference.to_numpy();pblend=data[best_blend_col].to_numpy();n=len(data)
gains=[]
for _ in range(N_BOOTSTRAP):
    idx=rng.integers(0,n,n)
    gains.append(log_loss(y[idx],pref[idx],labels=[0,1])-log_loss(y[idx],pblend[idx],labels=[0,1]))
gains=np.asarray(gains);ci=np.quantile(gains,[.025,.5,.975]);prob_gain=float(np.mean(gains>0))
bootstrap_summary=pd.Series({'best_blend':best_blend_row.model,'observed_gain':reference_loss-best_blend_row.logloss,
    'gain_ci95_lower':ci[0],'gain_bootstrap_median':ci[1],'gain_ci95_upper':ci[2],'bootstrap_probability_gain':prob_gain},name='resultado').to_frame()
display(bootstrap_summary);bootstrap_summary.to_csv(OUTPUT_DIR/'bootstrap_gain_summary.csv')
plt.figure(figsize=(7,4));sns.histplot(gains,bins=40);plt.axvline(0,color='crimson',ls='--');plt.xlabel('Ganancia de log-loss frente a referencia')
plt.tight_layout();plt.savefig(OUTPUT_DIR/'bootstrap_gain_distribution.png',dpi=170,bbox_inches='tight');plt.show()

## 11. Decisión final automática

In [ ]:
observed_gain=float(reference_loss-best_blend_row.logloss)
if observed_gain>=MIN_ACCEPTED_GAIN and ci[0]>0:
    status='accept_blend'
    conclusion='El ensamble mejora de forma consistente. Conservar ambos modelos y trasladar el peso a inferencia final.'
elif observed_gain>=MIN_ACCEPTED_GAIN and prob_gain>=.90:
    status='provisional_blend'
    conclusion='Existe una mejora pequeña pero plausible. Conservar como candidato secundario y no sustituir la referencia sin validación externa.'
else:
    status='keep_reference'
    conclusion='La CNN nueva no aporta una mejora suficientemente estable. Conservar el modelo de referencia y cerrar esta línea.'
decision={'status':status,'reference_file':str(REFERENCE_OOF.resolve()),'best_blend':str(best_blend_row.model),
          'reference_logloss':reference_loss,'blend_logloss':float(best_blend_row.logloss),'gain':observed_gain,
          'ci95_lower':float(ci[0]),'ci95_upper':float(ci[2]),'probability_gain':prob_gain,'conclusion':conclusion}
display(pd.Series(decision,name='resultado').to_frame());print(conclusion)
with open(OUTPUT_DIR/'FINAL_DECISION.json','w',encoding='utf-8') as f:json.dump(decision,f,ensure_ascii=False,indent=2)
data.to_csv(OUTPUT_DIR/'aligned_oof_and_cross_fitted_blends.csv',index=False)

## Cierre

El peso cruzado sirve para evaluar complementariedad sin optimizarlo sobre el mismo fold donde se mide. Si se acepta el ensamble, el peso operativo final puede estimarse con todos los OOF, pero debe interpretarse como parámetro para inferencia, no como una nueva métrica no sesgada.

No genere una entrega hasta comprobar que existen predicciones de inferencia de ambos modelos para exactamente los mismos UID requeridos por la plataforma.